# 숫자·영어·한국어 6방향 번역

숫자를 계산하는 규칙을 직접 작성하지 않고, 입력과 정답 쌍으로 번역 방법을 학습합니다.
**방향별 GRU + Attention 모델 6개**를 사용하며 영어↔한국어도 직접 번역합니다.
파일 구성과 자세한 실행 방법은 [README](README.md)를 참고하세요.

## 1. 설정

- `TRAIN = False`: 제공 모델을 불러와 평가합니다. 기본값입니다.
- `TRAIN = True`: 학습용 데이터로 새 모델을 학습하고 `new_models/`에 저장합니다.
- 숫자도 `"12345"`처럼 문자열로 처리합니다.
- 모든 구현은 이 노트북에 들어 있습니다.


In [1]:
import random
import math
import zipfile
from pathlib import Path
from collections import defaultdict
from types import SimpleNamespace

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence

TRAIN = False
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DIRECTIONS = [("num", "en"), ("en", "num"), ("num", "ko"),
              ("ko", "num"), ("en", "ko"), ("ko", "en")]

# 노트북과 데이터·모델을 같은 project01 폴더에 둡니다.
HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT = HERE if (HERE / "num2en.pt").is_file() else HERE / "project01"
DATA_PATH = PROJECT / "num2word.zip"
MODEL_DIR = PROJECT / "new_models" if TRAIN else PROJECT

config = SimpleNamespace(
    embedding_dim=128, hidden_dim=256, num_layers=1, dropout=0.1,
    attention=True, batch_size=256, learning_rate=0.001,
    max_epochs=30, patience=5, max_decode_steps=32, grad_clip=1.0, seed=SEED,
)

def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

torch.set_num_threads(4)
set_seed()
print("장치:", DEVICE, "| 학습:", TRAIN, "| 모델 폴더:", MODEL_DIR.name)


장치: cuda | 학습: False | 모델 폴더: project01


## 2. 데이터
숫자 키로 두 파일을 연결합니다. 행 번호로 연결하면 안 됩니다.
모든 방향에 같은 train/valid/test 분할을 적용하여 숫자 중복에 의한 누수를 막습니다.
자릿수와 두 파일의 공통 여부로 계층화합니다. 0의 고유 토큰을 위해 0은 train에 둡니다.


In [2]:
def normalize_text(text):
    return " ".join(text.strip().split())


def load_data(path):
    data = {}
    with zipfile.ZipFile(path) as archive:
        for lang, member in [("en", "num2txt.txt"), ("ko", "num2kor.txt")]:
            mapping = {}
            for line_number, line in enumerate(
                    archive.read(member).decode("utf-8-sig").splitlines(), 1):
                fields = line.split("\t")
                if len(fields) != 2:
                    raise ValueError(f"{member}:{line_number} 탭 구분 형식 오류")
                key, raw_text = fields
                text = normalize_text(raw_text)
                if not key or any(c not in "0123456789" for c in key) or not text:
                    raise ValueError(f"{member}:{line_number} 비어 있거나 잘못된 데이터")
                if key in mapping:
                    raise ValueError(f"{member}:{line_number} 중복 숫자 {key}")
                mapping[key] = text
            data[lang] = mapping
    return data


def split_number_keys(data, seed=SEED):
    groups = defaultdict(list)
    for key in sorted(set(data["en"]) | set(data["ko"])):
        membership = (key in data["en"], key in data["ko"])
        groups[(len(key), membership)].append(key)
    result = {"train": set(), "valid": set(), "test": set()}
    rng = random.Random(seed)
    for group in sorted(groups):
        keys = groups[group]
        rng.shuffle(keys)
        count = len(keys)
        holdout = max(1, round(count * 0.1)) if count >= 3 else 0
        result["valid"].update(keys[:holdout])
        result["test"].update(keys[holdout:2 * holdout])
        result["train"].update(keys[2 * holdout:])
    if "0" in set(data["en"]) | set(data["ko"]):
        result["valid"].discard("0")
        result["test"].discard("0")
        result["train"].add("0")
    assert not (result["train"] & result["valid"])
    assert not (result["train"] & result["test"])
    assert not (result["valid"] & result["test"])
    assert set.union(*result.values()) == set(data["en"]) | set(data["ko"])
    return result


def make_pairs(data, splits, source_lang, target_lang, split):
    available = set(splits[split])
    for lang in (source_lang, target_lang):
        if lang != "num":
            available.intersection_update(data[lang])
    rows = []
    for key in sorted(available, key=lambda key: (len(key), key)):
        source = key if source_lang == "num" else data[source_lang][key]
        target = key if target_lang == "num" else data[target_lang][key]
        rows.append((key, source, target))
    return rows


## 3. 토큰과 배치
숫자·한글은 문자, 영어는 단어 단위입니다.
PAD는 길이 맞추기, BOS는 출력 시작, EOS는 종료, UNK는 미등록 토큰입니다.


In [3]:
class Vocabulary:
    SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
    pad_id, bos_id, eos_id, unk_id = range(4)

    def __init__(self, language, tokens):
        self.language = language
        self.tokens = list(tokens)
        self.stoi = {token: idx for idx, token in enumerate(self.tokens)}
        assert self.tokens[:4] == self.SPECIALS
        assert len(self.tokens) == len(self.stoi)

    @staticmethod
    def tokenize(text, language):
        text = normalize_text(text)
        return text.split() if language == "en" else list(text)

    @classmethod
    def build(cls, language, texts):
        tokens = sorted({token for text in texts for token in cls.tokenize(text, language)})
        return cls(language, cls.SPECIALS + tokens)

    def encode(self, text, bos=False):
        ids = [self.stoi.get(t, self.unk_id) for t in self.tokenize(text, self.language)]
        return ([self.bos_id] if bos else []) + ids + [self.eos_id]

    def decode(self, ids):
        tokens = []
        for token_id in ids:
            token_id = int(token_id)  # 토큰 ID만 정수 처리; 입력 숫자 문자열은 변환하지 않는다.
            if token_id == self.eos_id:
                break
            if token_id in (self.pad_id, self.bos_id):
                continue
            tokens.append(self.tokens[token_id])
        separator = " " if self.language == "en" else ""
        return normalize_text(separator.join(tokens))

    def to_dict(self):
        return {"language": self.language, "tokens": self.tokens}

    def __len__(self):
        return len(self.tokens)


class TranslationDataset(Dataset):
    def __init__(self, rows, source_vocab, target_vocab):
        self.rows = rows
        self.encoded = [
            (torch.tensor(source_vocab.encode(source), dtype=torch.long),
             torch.tensor(target_vocab.encode(target, bos=True), dtype=torch.long))
            for _, source, target in rows]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        source, target = self.encoded[index]
        return source, target, self.rows[index]


def collate_fn(batch):
    source, target, rows = zip(*batch)
    lengths = torch.tensor([len(x) for x in source], dtype=torch.long)
    return (pad_sequence(source, batch_first=True, padding_value=0), lengths,
            pad_sequence(target, batch_first=True, padding_value=0), list(rows))


def make_loader(rows, source_vocab, target_vocab, config, shuffle=False):
    return DataLoader(TranslationDataset(rows, source_vocab, target_vocab),
                      batch_size=config.batch_size, shuffle=shuffle, num_workers=0,
                      collate_fn=collate_fn, pin_memory=DEVICE.type == "cuda",
                      generator=torch.Generator().manual_seed(config.seed))


## 4. 번역 모델

**인코더는 읽고, 디코더는 한 조각씩 출력합니다.**

1. Encoder GRU가 입력을 순서대로 읽고, 위치별 정보와 마지막 상태를 만듭니다.
2. Decoder GRU는 인코더의 마지막 상태를 받아 출력을 시작합니다.
3. Attention이 입력 위치별 중요도를 계산합니다. 위치별 정보를 그 비중으로 섞은 가중합을 디코더 정보와 결합하여 다음 토큰을 예측합니다. PAD 위치는 제외합니다.
4. 실제 생성에서는 출력 가능한 토큰 중 가장 높은 점수의 하나를 선택합니다(Greedy). EOS가 나오면 종료합니다.

학습할 때는 이전 **정답** 토큰을 주고 다음 토큰을 맞히게 합니다(teacher forcing).
실제 번역할 때는 이전 **예측** 토큰을 사용합니다.


In [4]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config.embedding_dim, padding_idx=0)
        self.dropout = nn.Dropout(config.dropout)
        self.gru = nn.GRU(config.embedding_dim, config.hidden_dim,
                          num_layers=config.num_layers, batch_first=True,
                          dropout=config.dropout if config.num_layers > 1 else 0)

    def forward(self, source, lengths):
        embedded = self.dropout(self.embedding(source))
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        output, hidden = self.gru(packed)
        output, _ = pad_packed_sequence(output, batch_first=True,
                                       total_length=source.size(1))
        return output, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.use_attention = config.attention
        self.hidden_dim = config.hidden_dim
        self.embedding = nn.Embedding(vocab_size, config.embedding_dim, padding_idx=0)
        self.dropout = nn.Dropout(config.dropout)
        self.gru = nn.GRU(config.embedding_dim, config.hidden_dim,
                          num_layers=config.num_layers, batch_first=True,
                          dropout=config.dropout if config.num_layers > 1 else 0)
        if self.use_attention:
            self.key_projection = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
            self.combine = nn.Linear(config.hidden_dim * 2, config.hidden_dim)
        self.output = nn.Linear(config.hidden_dim, vocab_size)

    def forward(self, token_ids, hidden, encoder_outputs, source_mask, keys=None):
        states, hidden = self.gru(self.dropout(self.embedding(token_ids)), hidden)
        if self.use_attention:
            if keys is None:
                keys = self.key_projection(encoder_outputs)
            scores = torch.bmm(states, keys.transpose(1, 2)) / math.sqrt(self.hidden_dim)
            scores = scores.masked_fill(~source_mask[:, None, :], float("-inf"))
            attention = scores.softmax(dim=-1)
            context = torch.bmm(attention, encoder_outputs)
            states = torch.tanh(self.combine(torch.cat((states, context), dim=-1)))
        return self.output(self.dropout(states)), hidden


class Seq2Seq(nn.Module):
    def __init__(self, source_size, target_size, config):
        super().__init__()
        self.encoder = Encoder(source_size, config)
        self.decoder = Decoder(target_size, config)

    def forward(self, source, lengths, decoder_input):
        encoded, hidden = self.encoder(source, lengths)
        logits, _ = self.decoder(decoder_input, hidden, encoded, source.ne(0))
        return logits

    @torch.inference_mode()
    def greedy_decode(self, source, lengths, max_steps):
        self.eval()
        encoded, hidden = self.encoder(source, lengths)
        mask = source.ne(0)
        keys = self.decoder.key_projection(encoded) if self.decoder.use_attention else None
        token = torch.full((source.size(0), 1), Vocabulary.bos_id,
                           dtype=torch.long, device=source.device)
        done = torch.zeros(source.size(0), dtype=torch.bool, device=source.device)
        generated = []
        for _ in range(max_steps):
            logits, hidden = self.decoder(token, hidden, encoded, mask, keys)
            scores = logits[:, -1, :]
            scores[:, Vocabulary.pad_id] = float("-inf")
            scores[:, Vocabulary.bos_id] = float("-inf")
            token = scores.argmax(dim=-1)
            token = torch.where(done, Vocabulary.eos_id, token)
            generated.append(token)
            done |= token.eq(Vocabulary.eos_id)
            token = token[:, None]
            if bool(done.all()):
                break
        return torch.stack(generated, dim=1)


## 5. 학습과 저장
정답을 한 칸 이동하여 다음 토큰을 예측합니다. PAD는 loss에서 제외합니다.
검증 loss가 가장 낮은 모델만 저장하고 개선이 멈추면 종료합니다.
기존 모델 파일 형식을 유지하여 이전에 학습한 가중치도 사용할 수 있습니다.


In [5]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, total_tokens = 0.0, 0

    with torch.set_grad_enabled(training):
        for source, lengths, target, _ in loader:
            source, target = source.to(DEVICE), target.to(DEVICE)
            logits = model(source, lengths, target[:, :-1])
            expected = target[:, 1:]
            loss = nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)), expected.reshape(-1),
                ignore_index=Vocabulary.pad_id, reduction="sum",
            )
            tokens = expected.ne(Vocabulary.pad_id).sum()
            if training:
                optimizer.zero_grad(set_to_none=True)
                (loss / tokens).backward()
                nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                optimizer.step()
            total_loss += loss.item()
            total_tokens += tokens.item()
    return total_loss / total_tokens


def save_model(path, model, source_vocab, target_vocab, epoch):
    torch.save({
        "format_version": 1,
        "model_state_dict": model.state_dict(),
        "source_vocab": source_vocab.to_dict(),
        "target_vocab": target_vocab.to_dict(),
        "config": vars(config),
        "metadata": {"best_epoch": epoch},
    }, path)


def load_model(path):
    saved = torch.load(path, map_location=DEVICE, weights_only=True)
    source_vocab = Vocabulary(**saved["source_vocab"])
    target_vocab = Vocabulary(**saved["target_vocab"])
    settings = SimpleNamespace(**saved["config"])
    model = Seq2Seq(len(source_vocab), len(target_vocab), settings).to(DEVICE)
    model.load_state_dict(saved["model_state_dict"])
    model.eval()
    return model, source_vocab, target_vocab, settings


def train_direction(data, splits, source_lang, target_lang):
    set_seed()
    train_rows = make_pairs(data, splits, source_lang, target_lang, "train")
    valid_rows = make_pairs(data, splits, source_lang, target_lang, "valid")
    source_vocab = Vocabulary.build(source_lang, (row[1] for row in train_rows))
    target_vocab = Vocabulary.build(target_lang, (row[2] for row in train_rows))
    train_loader = make_loader(train_rows, source_vocab, target_vocab, config, shuffle=True)
    valid_loader = make_loader(valid_rows, source_vocab, target_vocab, config)
    model = Seq2Seq(len(source_vocab), len(target_vocab), config).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    path = MODEL_DIR / f"{source_lang}2{target_lang}.pt"
    best_loss, stale_epochs = float("inf"), 0

    for epoch in range(1, config.max_epochs + 1):
        train_loss = run_epoch(model, train_loader, optimizer)
        valid_loss = run_epoch(model, valid_loader)
        if not math.isfinite(train_loss) or not math.isfinite(valid_loss):
            raise RuntimeError("학습 중 loss가 유한하지 않습니다.")
        print(f"{path.stem} epoch {epoch:02d}: train={train_loss:.4f}, valid={valid_loss:.4f}")
        if valid_loss < best_loss - 1e-5:
            best_loss, stale_epochs = valid_loss, 0
            save_model(path, model, source_vocab, target_vocab, epoch)
        else:
            stale_epochs += 1
        if stale_epochs >= config.patience:
            break


## 6. 평가
정답을 주지 않고 생성합니다. 문자열 전체 일치와 EOS 종료를 모두 요구합니다.
자릿수별 표본 수를 함께 봅니다. 영어↔한글은 9~10자리 데이터가 없으며 8자리 test도 1개뿐입니다.


In [6]:
@torch.inference_mode()
def predict_rows(model, loader, target_vocab, max_steps):
    records = []
    for source, lengths, _, rows in loader:
        output = model.greedy_decode(source.to(DEVICE), lengths, max_steps).cpu().tolist()
        for (key, text, reference), ids in zip(rows, output):
            prediction = target_vocab.decode(ids)
            terminated = Vocabulary.eos_id in ids
            # 종료하지 못한 잘린 출력은 문자열이 우연히 같아도 오답이다.
            records.append({"number": key, "digits": len(key), "source": text,
                            "reference": reference, "prediction": prediction,
                            "terminated": terminated,
                            "correct": terminated and prediction == reference})
    return records


def summarize_predictions(records):
    by_digits = {}
    for digits in range(1, 11):
        rows = [row for row in records if row["digits"] == digits]
        correct = sum(row["correct"] for row in rows)
        by_digits[str(digits)] = {"count": len(rows), "correct": correct,
                                  "exact_match": correct / len(rows) if rows else None}
    return {"count": len(records), "correct": sum(row["correct"] for row in records),
            "exact_match": sum(row["correct"] for row in records) / len(records) if records else None,
            "not_terminated": sum(not row["terminated"] for row in records),
            "by_digits": by_digits}


def evaluate_direction(data, splits, source_lang, target_lang):
    name = f"{source_lang}2{target_lang}"
    model, source_vocab, target_vocab, settings = load_model(MODEL_DIR / f"{name}.pt")
    rows = make_pairs(data, splits, source_lang, target_lang, "test")
    loader = make_loader(rows, source_vocab, target_vocab, settings)
    records = predict_rows(model, loader, target_vocab, settings.max_decode_steps)
    result = summarize_predictions(records)
    print(f"{name}: {result['exact_match']:.2%} ({result['correct']}/{result['count']})")
    for digits, stats in result["by_digits"].items():
        score = f"{stats['exact_match']:.1%}" if stats["count"] else "평가 불가"
        print(f"  {digits}자리: {score}, N={stats['count']}")
    for row in [r for r in records if not r["correct"]][:3]:
        print(f"  오답: {row['source']} → {row['prediction']} / 정답: {row['reference']}")
    return result


## 7. 실행
TRAIN=False이면 새 학습 없이 기존 모델을 전체 test로 다시 평가합니다.
TRAIN=True이면 별도 new_models 폴더에 새 모델을 저장한 뒤 평가합니다.
설정과 함수 정의 뒤 학습·평가 실행을 이 셀 한 곳에 모았습니다.


In [7]:
data = load_data(DATA_PATH)
splits = split_number_keys(data)
if TRAIN:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for source, target in DIRECTIONS:
        train_direction(data, splits, source, target)

results = {}
for source, target in DIRECTIONS:
    results[f"{source}2{target}"] = evaluate_direction(data, splits, source, target)


num2en: 99.90% (9991/10001)
  1자리: 100.0%, N=1
  2자리: 100.0%, N=9
  3자리: 100.0%, N=90
  4자리: 100.0%, N=900
  5자리: 100.0%, N=1500
  6자리: 100.0%, N=1500
  7자리: 99.9%, N=1500
  8자리: 99.9%, N=1501
  9자리: 99.7%, N=1500
  10자리: 99.8%, N=1500
  오답: 6000840 → six million eight hundred ninety six / 정답: six million eight hundred forty
  오답: 9000536 → nine million five hundred ninety six / 정답: nine million five hundred thirty six
  오답: 63000247 → sixty three million two hundred sixty six / 정답: sixty three million two hundred forty seven
en2num: 99.95% (9996/10001)
  1자리: 100.0%, N=1
  2자리: 100.0%, N=9
  3자리: 100.0%, N=90
  4자리: 100.0%, N=900
  5자리: 100.0%, N=1500
  6자리: 100.0%, N=1500
  7자리: 99.9%, N=1500
  8자리: 99.9%, N=1501
  9자리: 99.9%, N=1500
  10자리: 99.9%, N=1500
  오답: nine million five hundred thirty six → 90000536 / 정답: 9000536
  오답: eighty three million one hundred forty thousand → 831400000 / 정답: 83140000
  오답: one hundred ninety six million five hundred fifty two → 196000525 / 정답: 19600

## 8. 저장 모델로 사용하기

마지막 코드 셀에서 모델 파일을 다시 읽어 6방향 번역을 실행합니다.
`translate(입력 문자열, 입력 언어, 출력 언어)` 형태로 사용하며 언어 코드는 `num`, `en`, `ko`입니다.

```python
translate("12345", "num", "ko")
```

번역만 실행하려면 `TRAIN = False`로 두고 1~6절 코드 셀을 실행한 뒤 7절을 건너뛰어 이 셀을 실행하세요.
이 경우 데이터 ZIP 없이 모델 파일만으로 사용할 수 있습니다. 각 호출은 해당 모델을 새로 불러옵니다.
숫자를 포함한 입력은 반드시 문자열이어야 합니다. 빈 입력, 사전에 없는 토큰, 잘못된 숫자 형식은 오류로 알립니다.


In [8]:
def translate(text, source="num", target="ko"):
    if not isinstance(text, str):
        raise TypeError('입력은 문자열이어야 합니다. 예: "12345"')
    if (source, target) not in DIRECTIONS:
        raise ValueError("지원하지 않는 변환 방향입니다.")
    text = normalize_text(text)
    if not text:
        raise ValueError("입력이 비어 있습니다.")
    if source == "num" and any(c not in "0123456789" for c in text):
        raise ValueError("숫자는 0~9 문자만 입력하세요.")
    model, source_vocab, target_vocab, settings = load_model(MODEL_DIR / f"{source}2{target}.pt")
    ids = source_vocab.encode(text)
    if Vocabulary.unk_id in ids:
        raise ValueError("학습 사전에 없는 입력 토큰이 있습니다.")
    encoded = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    lengths = torch.tensor([len(ids)])
    output = model.greedy_decode(encoded, lengths, settings.max_decode_steps)[0].tolist()
    if Vocabulary.eos_id not in output:
        raise RuntimeError("최대 출력 길이 안에 종료하지 못했습니다.")
    return target_vocab.decode(output)

examples = {"num": "12345", "en": "twelve thousand three hundred forty five",
            "ko": "만 이천삼백사십오"}
for source, target in DIRECTIONS:
    print(f"{source} → {target}: {translate(examples[source], source, target)}")


num → en: twelve thousand three hundred forty five
en → num: 12345
num → ko: 만 이천삼백사십오
ko → num: 12345
en → ko: 만 이천삼백사십오
ko → en: twelve thousand three hundred forty five
